## Task 1: Understanding Chat Messages

SystemMessage: Defines the AI's behavior and instructions.


HumanMessage: Represents the user's input.


AIMessage: Represents the AI's response.

Message-based prompting is better because it separates instructions, user questions, and AI responses, making conversations more structured and easier to manage.

## Task 2: MessagePlaceholder Usage

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Create a chat prompt template with system instructions, history, and current input
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful AI assistant.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

history = [
    HumanMessage(content="My name is Manish."),
    AIMessage(content="Hello Manish! Nice to meet you."),
    HumanMessage(content="I am learning Python."),
    AIMessage(content="That's great! Python is useful for AI and ML.")
]

# Initialize Groq
llm = ChatGroq(model='openai/gpt-oss-120b')

# Initialize the output parser
parser = StrOutputParser()

# Create a chain
chain = prompt | llm | parser

# Invoke the chain with the conversation history and current question
response = chain.invoke({
    'history': history,
    'input': 'What am I learning?'
})

# Print the response
print(response)

You’re learning **Python**, the versatile, high‑level programming language that’s widely used for everything from web development and automation to data analysis, scientific computing, and artificial‑intelligence projects. Happy coding!


## Task 3: Basic Message History

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Store chat history for different sessions
store = {}

def get_history(session_id: str):
    # Create history if the session does not exist
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Using the prev task prompt, llm and parser
chain = prompt | llm | parser

# Automatically manage conversation history
chat_with_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key='input',
    history_messages_key='history'
)

response1 = chat_with_history.invoke({
    'input': 'What is Python?'
},
config={'configurable': {'session_id': 'manish123'}} 
)

response2 = chat_with_history.invoke({
    'input': 'Give me an example.'
},
config={'configurable': {'session_id': 'manish123'}} 
)

# Print both responses
print("response1", response1)
print("\nresponse2", response2)

print("\nChat History:", get_history("manish123").messages)

## Task 4: Trimming Chat History

In [ ]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Initialize Groq
llm = ChatGroq(model='openai/gpt-oss-120b')

# Initialize the output parser
parser = StrOutputParser()

# Trim older messages to limit history
trimmer = trim_messages(max_tokens=2000, token_counter=llm, include_system=True, allow_partial=False, start_on='human')

# Create a chat prompt template with system instructions, history, and current input
prompt = ChatPromptTemplate.from_messages([
	('system', 'You are helpful AI Assistant with Specialized in Machine Learning, You will be having chat past history So always use to continue the chat naturally and answer in short as possible'),
    MessagesPlaceholder(variable_name='history'),
	('human', '{input}')
])

chain = (RunnablePassthrough.assign(history=itemgetter('history') | trimmer) | prompt | llm | parser) 

# Store conversation history
history = []

def chat(user_input):
    # Generate response using trimmed history
    response = chain.invoke({
        'history': history,
        'input': user_input
	})

    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))

    return response

# Test the chatbot
print("AI:", chat("What is Python?"))
print("AI:", chat("What is machine learning?"))
print("AI:", chat("What did we discuss earlier?"))

## Task 5: Build Q&A Chatbot

In [23]:
from langchain_core.messages import HumanMessage, AIMessage

# Store conversation history
history = []

while True:
    user_input = input('You: ')

    if user_input.lower() == 'exit':
        print('Chat Ended...')
        break

    # using task 4 chain
    response = chain.invoke({
        'history': history,
        'input': user_input
    })

    print("AI:", response)

    # Save user and AI messages
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))

AI: Hello! How can I help you today?
AI: Nice to meet you, Manish! What would you like to discuss today?
AI: Sure! Using a = 1, b = 2 … z = 26 (case‑insensitive):

- M = 13  
- A = 1  
- N = 14  
- I = 9  
- S = 19  
- H = 8  

**Total = 13 + 1 + 14 + 9 + 19 + 8 = 64**.
Chat Ended...


## Task 6: Build Stateful Chatbot Application

In [ ]:
## Please refer app.py for streamlit project

## Task 7: Observations & Insights

1. Why is chat history important?
   Chat history helps the chatbot understand context, remember previous questions, and answer follow-up questions.

2. Trade-offs between long memory and performance:
   Long history provides more context but increases token usage, cost, and response time.

3. When to summarize vs. trim history:
   Summarize history when older information remains important. Trim history when older messages are no longer needed.
   
4. Difference between message placeholders and memory:
   MessagesPlaceholder inserts conversation messages into a prompt. Memory manages how conversation history is stored, retrieved, and maintained.